# 11. 텍스트의 유사도 (Text Similarity)

이번 장에서는 컴퓨터가 텍스트의 **유사도(similarity)**를 어떻게 계산하는지를 다룬다. 두 문서가 얼마나 비슷한지를 수치로 재려면, 먼저 텍스트를 **숫자(벡터)로 표현**해야 한다. 그래서 단어를 수치화하는 가장 기초적인 방법들인 **Bag of Words(BoW)**, **문서 단어 행렬(DTM)**, **TF-IDF**를 차례로 배우고, 이렇게 만든 벡터로 문서 간 거리를 재는 **코사인 유사도**, **유클리드 거리**, **자카드 유사도**까지 정리한다.

> **이 장의 흐름** : 단어의 표현 방법(국소 vs 분산) → Bag of Words → 문서 단어 행렬(DTM)과 그 한계 → TF-IDF로 가중치 주기 → 코사인 유사도와 추천 시스템 → 유클리드 거리·자카드 유사도.

> 이 장은 "문서를 벡터로 바꾼 뒤, 벡터 간 유사도를 잰다"는 하나의 큰 줄기를 따라간다. 핵심 개념마다 짧은 파이썬 코드(순수 구현 + 사이킷런)로 교안의 숫자를 직접 재현하며 손으로 따라가 본다.

# 1. 단어의 표현 방법 (Word Representation)

텍스트의 유사도를 구하려면 단어를 숫자로 표현해야 한다. 이 절에서는 단어를 표현하는 방법에 크게 어떤 갈래가 있는지, 그리고 이 책에서 어떤 순서로 단어 표현을 배워 나갈지를 먼저 짚는다.

## 1-1. 국소 표현과 분산 표현

단어의 표현 방법은 크게 **국소 표현(Local Representation)**과 **분산 표현(Distributed Representation)**으로 나뉜다.

- **국소 표현** : 해당 단어 **그 자체만 보고** 특정 값을 맵핑하여 표현한다. 예를 들어 puppy=1, cute=2, lovely=3처럼 단어마다 번호를 붙이는 방식이다.
- **분산 표현** : 그 단어를 표현하기 위해 **주변 단어를 참고**한다. puppy 근처에 cute, lovely가 자주 등장하므로 "puppy는 cute, lovely한 느낌"이라고 정의하는 방식이다.

이 둘의 결정적 차이는 **뉘앙스(의미) 표현 여부**다. 국소 표현은 단어의 의미·뉘앙스를 담지 못하지만, 분산 표현은 주변 맥락을 끌어오므로 단어의 뉘앙스를 표현할 수 있다.

> **용어 정리** : 국소 표현(Local Representation)을 **이산 표현(Discrete Representation)**, 분산 표현(Distributed Representation)을 **연속 표현(Continuous Representation)**이라고도 부른다. 구글의 토마스 미코로브는 LSA·LDA처럼 의미를 담는 방법은 연속 표현이지만 Word2Vec 같은 분산 표현과는 접근이 다르다고 보아, 연속 표현을 분산 표현보다 더 큰 개념으로 설명하기도 했다.

## 1-2. 단어 표현의 카테고리화

이 책에서 단어 표현을 분류하는 큰 그림은 다음과 같다.

```
Word Representation
├─ Local Representation (국소 / 이산 표현)
│   ├─ One-hot Vector
│   ├─ N-gram
│   └─ Count Based ─ Bag of Words (DTM)
└─ Continuous Representation (연속 표현)
    ├─ Prediction Based ─ Word2Vec (FastText)
    └─ Count Based
        ├─ Full Document ─ LSA
        └─ Windows ─ GloVe
```

**이번 장에서 다루는 것**은 왼쪽 갈래인 국소 표현 중 **Count Based**에 속한다. 단어의 **빈도수(frequency)를 카운트**하여 수치화하는 **Bag of Words**, 그 확장인 **DTM(또는 TDM)**, 그리고 단어의 중요도에 따라 가중치를 주는 **TF-IDF**를 배운다.

오른쪽의 **연속 표현**(예측 기반의 Word2Vec·FastText, 카운트 기반의 LSA·GloVe)은 다음 장의 워드 임베딩에서 본격적으로 다룬다. 지금은 "빈도수를 세서 단어를 벡터로 만든다"는 카운트 기반 접근에 집중한다.

# 2. Bag of Words (BoW)

**Bag of Words(BoW)**는 단어의 **등장 순서를 전혀 고려하지 않고**, 단어들의 **출현 빈도에만 집중**하는 텍스트 수치화 방법이다. 이 절에서는 BoW의 개념과 직접 구현, 그리고 사이킷런으로 손쉽게 만드는 방법까지 살펴본다.

## 2-1. Bag of Words란?

BoW를 직역하면 "단어들의 가방"이다. 문서에 있는 단어들을 모두 가방에 넣고 흔들어 섞는다고 상상해 보자. 특정 단어가 N번 등장했다면 가방 안에 그 단어가 N개 들어 있게 되고, 흔들어 섞었으니 **순서는 더 이상 중요하지 않다.**

BoW를 만드는 과정은 두 단계다.

1. 각 단어에 **고유한 정수 인덱스**를 부여한다 (단어 집합, vocabulary 생성).
2. 각 인덱스 위치에 해당 단어 토큰의 **등장 횟수**를 기록한 벡터를 만든다.

아래 문서로 BoW를 직접 만들어 본다.

> 문서1 : 정부가 발표하는 물가상승률과 소비자가 느끼는 물가상승률은 다르다.

## 2-2. BoW 직접 만들어 보기

한국어는 조사가 붙기 때문에 띄어쓰기만으로 토큰화하면 안 되고, **형태소 분석기**(예: KoNLPy의 Okt)로 토큰화해야 제대로 된 BoW가 나온다. 아래 코드는 형태소 분석 결과(토큰 리스트)를 입력으로 받아 단어 집합과 BoW 벡터를 만든다.

> 외부 라이브러리 없이 동작하도록, `okt.morphs(문서1)`가 만들어 내는 토큰 리스트를 직접 적어 두었다. 실제 환경에서는 `from konlpy.tag import Okt`로 Okt를 불러와 `okt.morphs(document)`를 호출하면 같은 결과를 얻는다.

In [ ]:
def build_bag_of_words(tokenized_document):
    word_to_index = {}
    bow = []
    for word in tokenized_document:
        if word not in word_to_index:
            # 새 단어: 다음 인덱스를 부여하고, 빈도 1로 시작
            word_to_index[word] = len(word_to_index)
            bow.insert(len(word_to_index) - 1, 1)
        else:
            # 재등장: 해당 인덱스 위치의 빈도를 1 증가
            index = word_to_index[word]
            bow[index] = bow[index] + 1
    return word_to_index, bow

# 문서1을 Okt로 형태소 분석한 결과 (okt.morphs(doc1)와 동일)
doc1_tokens = ['정부', '가', '발표', '하는', '물가상승률', '과',
               '소비자', '가', '느끼는', '물가상승률', '은', '다르다']

vocab, bow = build_bag_of_words(doc1_tokens)
print('vocabulary       :', vocab)
print('bag of words vec :', bow)
# 인덱스 4(물가상승률)가 두 번 등장 -> 값이 2

BoW 벡터를 보면 인덱스 4에 해당하는 **물가상승률**이 두 번 언급되어 값이 **2**이고, 나머지는 1이다(인덱스는 0부터 시작한다). BoW는 "어떤 단어가 얼마나 등장했는가"를 수치화하므로, 주로 **문서 분류**나 **문서 간 유사도** 문제에 쓰인다. 가령 '달리기·체력·근력'이 자주 나오면 체육 문서, '미분·방정식·부등식'이 자주 나오면 수학 문서로 분류할 수 있다.

## 2-3. CountVectorizer로 BoW 만들기

사이킷런의 **CountVectorizer**는 단어 빈도를 세어 벡터로 만들어 준다. 영어라면 손쉽게 BoW를 만들 수 있다.

다만 주의할 점이 둘 있다. 첫째, CountVectorizer는 기본적으로 **길이가 2 이상인 토큰만** 인식하므로 알파벳 한 글자 `I`는 사라진다. 둘째, **띄어쓰기 기준으로만** 자르는 낮은 수준의 토큰화를 하므로, 한국어에 적용하면 '물가상승률과'와 '물가상승률은'을 서로 다른 단어로 인식해 BoW가 제대로 만들어지지 않는다.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = ['you know I want your love. because I love you.']
vector = CountVectorizer()

# 코퍼스로부터 각 단어의 빈도수를 기록
print('bag of words vec :', vector.fit_transform(corpus).toarray())
# 각 단어에 부여된 인덱스
print('vocabulary       :', vector.vocabulary_)
# you, love가 두 번씩 등장 -> 각 인덱스에서 값 2 / 한 글자 'I'는 제거됨

## 2-4. 불용어를 제거한 BoW

**불용어(stopwords)**는 자연어 처리에서 큰 의미를 갖지 않는 단어들이다. BoW는 어떤 단어가 자주 등장하는지를 보는 방법이므로, 불용어를 제거하면 더 정제된(정확도 높은) BoW를 만들 수 있다. CountVectorizer는 불용어 제거 기능을 지원한다.

방법은 셋이다 — ① 사용자가 직접 정의한 리스트, ② CountVectorizer가 제공하는 영어 불용어(`stop_words="english"`), ③ NLTK가 제공하는 불용어. 아래는 세 방법을 모두 보여 준다.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

text = ["Family is not an important thing. It's everything."]

# 1) 사용자가 직접 정의한 불용어
vect = CountVectorizer(stop_words=["the", "a", "an", "is", "not"])
print('[직접 정의] vec  :', vect.fit_transform(text).toarray())
print('[직접 정의] vocab:', vect.vocabulary_)

# 2) CountVectorizer가 제공하는 영어 불용어
vect = CountVectorizer(stop_words="english")
print('[내장 영어] vec  :', vect.fit_transform(text).toarray())
print('[내장 영어] vocab:', vect.vocabulary_)

# 3) NLTK 불용어가 필요하면 아래처럼 쓸 수 있다(별도 설치/다운로드 필요)
#   from nltk.corpus import stopwords
#   stop_words = stopwords.words("english")
#   vect = CountVectorizer(stop_words=stop_words)

# 3. DTM과 TF-IDF

여러 문서의 BoW를 하나로 합치면 문서들을 서로 비교할 수 있게 된다. 이 절에서는 여러 문서의 BoW를 행렬로 묶은 **문서 단어 행렬(DTM)**, 그 한계, 그리고 단어 중요도에 가중치를 주는 **TF-IDF**를 배운다.

## 3-1. 문서 단어 행렬 (DTM)

**문서 단어 행렬(Document-Term Matrix, DTM)**은 다수의 문서에서 등장하는 각 단어의 빈도를 행렬로 표현한 것이다. 쉽게 말해 **각 문서의 BoW를 하나의 행렬로 쌓은 것**이다(행과 열을 바꾸면 TDM이라 부른다). 다음 네 문서를 띄어쓰기 기준으로 토큰화했다고 하자.

> 문서1 : 먹고 싶은 사과
> 문서2 : 먹고 싶은 바나나
> 문서3 : 길고 노란 바나나 바나나
> 문서4 : 저는 과일이 좋아요

이를 DTM으로 표현하면 다음과 같다.

| | 과일이 | 길고 | 노란 | 먹고 | 바나나 | 사과 | 싶은 | 저는 | 좋아요 |
|:--|:--:|:--:|:--:|:--:|:--:|:--:|:--:|:--:|:--:|
| **문서1** | 0 | 0 | 0 | 1 | 0 | 1 | 1 | 0 | 0 |
| **문서2** | 0 | 0 | 0 | 1 | 1 | 0 | 1 | 0 | 0 |
| **문서3** | 0 | 1 | 1 | 0 | 2 | 0 | 0 | 0 | 0 |
| **문서4** | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 1 |

각 칸은 해당 문서에서 그 단어가 등장한 빈도다. 이렇게 문서를 수치화하면 문서들을 서로 비교할 수 있게 된다는 점에 DTM의 의의가 있다.

## 3-2. DTM의 한계

DTM은 간단하지만 본질적 한계가 둘 있다.

**한계 1 — 희소 표현(Sparse Representation)** : 각 문서 벡터의 차원은 **전체 단어 집합의 크기**가 된다. 코퍼스가 방대하면 차원이 수만 이상으로 커지고, 대부분의 값이 0인 **희소 벡터(sparse vector)**가 된다. 위 예제만 봐도 0이 아닌 값보다 0이 훨씬 많다. 희소 벡터는 저장 공간과 계산량을 크게 늘리므로, 전처리(구두점·저빈도어·불용어 제거, 어간/표제어 추출)로 단어 집합 크기를 줄이는 일이 중요하다.

**한계 2 — 단순 빈도수 기반** : 빈도만 세는 방식은 한계가 있다. 영어의 불용어 `the`는 어느 문서에나 자주 등장한다. 그런데 문서1·2·3에서 `the`의 빈도가 똑같이 높다고 해서 이 문서들이 유사하다고 판단해선 안 된다. **중요한 단어와 불필요한 단어에 가중치를 다르게** 줄 방법이 필요하다. 이 아이디어가 바로 **TF-IDF**다.

## 3-3. TF-IDF (단어 빈도-역 문서 빈도)

**TF-IDF(Term Frequency-Inverse Document Frequency)**는 단어의 빈도(TF)와 역 문서 빈도(IDF)를 곱해, DTM 내 각 단어에 **중요도 가중치**를 주는 방법이다. 문서를 $d$, 단어를 $t$, 총 문서 수를 $n$ 이라 하면 다음과 같이 정의한다.

**1) $\text{tf}(d, t)$ — 특정 문서 $d$ 에서 단어 $t$ 의 등장 횟수.** 사실 앞서 만든 DTM의 각 칸 값이 곧 TF다.

**2) $\text{df}(t)$ — 단어 $t$ 가 등장한 문서의 수.** 한 문서에서 몇 번 나왔는지는 보지 않고, **몇 개의 문서에 나왔는지**만 센다. 위 DTM에서 '바나나'는 문서2와 문서3에 등장했으므로 df는 2다(문서3에서 두 번 나와도 df는 그대로 2).

**3) $\text{idf}(t)$ — df에 반비례하는 값.**

$$ \text{idf}(t) = \log\left(\frac{n}{1 + \text{df}(t)}\right) $$

> **왜 log를 씌우나** : 단순히 $n/\text{df}(t)$ 로 쓰면 $n$ 이 커질수록 IDF가 기하급수적으로 폭증한다. 불용어는 희귀어보다 수십~수백 배 자주 등장하는데, log가 없으면 희귀어에 엄청난 가중치가 붙는다. log는 이 격차를 완만하게 줄여 준다. 분모에 **1을 더하는** 이유는 어떤 단어가 전체 문서에 한 번도 안 나와 df=0일 때 분모가 0이 되는 것을 막기 위함이다.

TF-IDF의 직관은 명확하다. **모든 문서에 흔한 단어는 중요도가 낮고(IDF가 작다), 특정 문서에만 자주 나오는 단어는 중요도가 높다.** 그래서 `the`, `a` 같은 불용어는 자연스럽게 TF-IDF 값이 낮아진다.

## 3-4. 파이썬으로 TF-IDF 직접 구현하기

3-1의 네 문서를 그대로 써서 TF, IDF, TF-IDF를 손으로 구현해 본다. 로그는 자연 로그(밑이 $e$ 인 ln)를 사용한다. 먼저 단어 집합과 함수를 정의하고 DTM(=TF)을 만든다.

In [ ]:
import pandas as pd
from math import log

docs = [
    '먹고 싶은 사과',
    '먹고 싶은 바나나',
    '길고 노란 바나나 바나나',
    '저는 과일이 좋아요',
]

# 단어 집합(정렬)
vocab = sorted({w for doc in docs for w in doc.split()})
N = len(docs)   # 총 문서 수

def tf(t, d):
    return d.count(t)

def idf(t):
    df = sum(1 for doc in docs if t in doc)   # t가 등장한 문서 수
    return log(N / (df + 1))

def tfidf(t, d):
    return tf(t, d) * idf(t)

# DTM(=TF) 만들기
dtm = [[tf(t, d) for t in vocab] for d in docs]
tf_ = pd.DataFrame(dtm, columns=vocab)
print('[DTM = TF]')
print(tf_)

이제 각 단어의 **IDF**를 구한다. 문서1개에만 등장한 단어(IDF≈0.693)와 문서2개에 등장한 단어(IDF≈0.288)가 값에서 차이를 보인다 — IDF가 여러 문서에 걸친 흔한 단어의 가중치를 낮춰 주기 때문이다.

In [ ]:
idf_ = pd.DataFrame([idf(t) for t in vocab], index=vocab, columns=['IDF'])
print('[IDF]')
print(idf_)

마지막으로 **TF-IDF 행렬**을 구한다. TF에 단어별 IDF를 곱한 값이다. 같은 '바나나'라도 문서2(TF=1)보다 문서3(TF=2)에서 TF-IDF가 더 큰데, 이는 **특정 문서에서 자주 등장한 단어를 그 문서에서 더 중요하다고 판단**하는 TF-IDF의 관점을 그대로 보여 준다.

In [ ]:
tfidf_ = pd.DataFrame([[tfidf(t, d) for t in vocab] for d in docs], columns=vocab)
print('[TF-IDF]')
print(tfidf_.round(6))

## 3-5. 사이킷런으로 DTM과 TF-IDF 만들기

실무에서는 직접 구현하는 대신 사이킷런을 쓴다. **CountVectorizer**로 DTM을, **TfidfVectorizer**로 TF-IDF를 자동 계산한다.

> 사이킷런의 TF-IDF는 위에서 배운 기본식을 **조정한 식**을 쓴다. 기본식은 $\text{df}(t) = n - 1$ 같은 경우 분자와 분모가 같아져 $\log 1 = 0$, 즉 IDF가 0이 되어 가중치 역할을 못 하는 문제가 있다. 사이킷런은 로그항 분자에 1을 더하고, 로그항 전체에 1을 더하며, 결과에 **L2 정규화**를 적용하는 식으로 이를 보완한다. 의도(흔한 단어는 낮게, 희귀한 단어는 높게)는 그대로다.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

corpus = [
    'you know I want your love',
    'I like you',
    'what should I do ',
]

# (1) CountVectorizer -> DTM
cv = CountVectorizer()
print('[DTM]')
print(cv.fit_transform(corpus).toarray())
print('vocab:', cv.vocabulary_)
print()

# (2) TfidfVectorizer -> TF-IDF (L2 정규화된 값)
tv = TfidfVectorizer().fit(corpus)
print('[TF-IDF]')
print(tv.transform(corpus).toarray().round(4))
print('vocab:', tv.vocabulary_)

# 4. 코사인 유사도와 추천 시스템

DTM·TF-IDF로 문서를 벡터로 만들었으니, 이제 벡터 간 유사도를 잴 차례다. 가장 널리 쓰이는 **코사인 유사도**를 배우고, 이를 이용한 **영화 추천 시스템**을 직접 만들어 본다.

## 4-1. 코사인 유사도 (Cosine Similarity)

**코사인 유사도**는 두 벡터가 이루는 **각도**로 유사도를 잰다. 두 벡터의 방향이 완전히 같으면 1, 90°면 0, 180°(정반대)면 -1이다. 즉 **-1 이상 1 이하**의 값을 가지며, **1에 가까울수록 유사**하다. 직관적으로는 두 벡터가 가리키는 방향이 얼마나 비슷한지를 뜻한다.

두 벡터 $A$, $B$ 에 대해 코사인 유사도는 다음과 같다.

$$ \text{similarity} = \cos(\theta) = \frac{A \cdot B}{\|A\|\,\|B\|} = \frac{\sum_{i=1}^{n} A_i \times B_i}{\sqrt{\sum_{i=1}^{n}(A_i)^2} \times \sqrt{\sum_{i=1}^{n}(B_i)^2}} $$

다음 세 문서의 DTM으로 코사인 유사도를 구해 본다.

> 문서1 : 저는 사과 좋아요
> 문서2 : 저는 바나나 좋아요
> 문서3 : 저는 바나나 좋아요 저는 바나나 좋아요

| | 바나나 | 사과 | 저는 | 좋아요 |
|:--|:--:|:--:|:--:|:--:|
| **문서1** | 0 | 1 | 1 | 1 |
| **문서2** | 1 | 0 | 1 | 1 |
| **문서3** | 2 | 0 | 2 | 2 |

In [ ]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

doc1 = np.array([0, 1, 1, 1])
doc2 = np.array([1, 0, 1, 1])
doc3 = np.array([2, 0, 2, 2])   # 문서2의 모든 빈도를 2배로 늘린 문서

print('문서1 vs 문서2 :', round(cos_sim(doc1, doc2), 2))
print('문서1 vs 문서3 :', round(cos_sim(doc1, doc3), 2))
print('문서2 vs 문서3 :', round(cos_sim(doc2, doc3), 2))   # 방향이 같으므로 1.00

## 4-2. 코사인 유사도의 장점: 문서 길이에 강건함

위 결과에서 눈여겨볼 점은 **문서2와 문서3의 유사도가 1**이라는 것이다. 문서3은 문서2에서 **모든 단어의 빈도가 똑같이 2배**로 늘었을 뿐이다. 즉, 한 문서의 모든 단어 빈도가 동일 비율로 커지면 원래 문서와 코사인 유사도가 1이 된다.

이것이 코사인 유사도의 강점이다. 같은 주제의 문서 B가 문서 A의 두 배 길이라면, **유클리드 거리**로는 길이 차이 때문에 엉뚱한 결과(다른 주제의 C와 더 가깝다고 판단)가 나올 수 있다. 코사인 유사도는 벡터의 **방향(패턴)**에만 집중하므로, 문서 길이가 달라도 비교적 공정하게 유사도를 잰다.

## 4-3. 유사도 기반 추천 시스템 구현하기

이제 **TF-IDF + 코사인 유사도**만으로 줄거리 기반 영화 추천 시스템을 만든다. 실제로는 캐글의 영화 데이터셋(약 4.5만 개 영화의 `title`, `overview`)을 쓰지만, 여기서는 외부 다운로드 없이 동작하도록 **작은 장난감 데이터**로 메커니즘을 그대로 재현한다(실데이터에서도 코드는 동일하다).

흐름은 셋이다 — ① `overview`로 TF-IDF 행렬을 만들고, ② 영화 간 코사인 유사도 행렬을 구한 뒤, ③ 제목으로 가장 비슷한 영화들을 찾는다.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 장난감 영화 데이터 (제목 + 줄거리). 우주/로맨스/요리 세 묶음으로 구성
data = pd.DataFrame({
    'title': ['Space Odyssey', 'Galaxy Quest', 'Star Voyage',
              'Love in Paris', 'Romance Cafe', 'Cooking Master'],
    'overview': [
        'astronauts travel through space to a distant planet aboard a starship',
        'a starship crew explores the galaxy and fights aliens in deep space',
        'a space crew journeys across the galaxy aboard a starship to a planet',
        'two strangers fall in love during a romantic trip to the city of paris',
        'a couple fall in love at a small cafe in a romantic european city',
        'a chef competes to cook the best dish in a famous cooking contest',
    ],
}).copy()

# overview에 결측값이 있으면 TF-IDF에서 에러가 나므로 빈 문자열로 대체
data['overview'] = data['overview'].fillna('')

# (1) TF-IDF 행렬 (영어 불용어 제거)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['overview'])
print('TF-IDF 행렬 크기:', tfidf_matrix.shape)   # (영화 수, 단어 수)

# (2) 영화 간 코사인 유사도 행렬
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print('코사인 유사도 행렬 크기:', cosine_sim.shape)

In [ ]:
# 제목 -> 인덱스 매핑
title_to_index = dict(zip(data['title'], data.index))

def get_recommendations(title, cosine_sim=cosine_sim):
    idx = title_to_index[title]                                    # 입력 영화의 인덱스
    sim_scores = list(enumerate(cosine_sim[idx]))                  # 모든 영화와의 유사도
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:3]                                   # 자기 자신 제외, 상위 2개
    movie_indices = [i for i, score in sim_scores]
    return data['title'].iloc[movie_indices]

# 'Star Voyage'와 줄거리가 비슷한 영화 추천 -> 우주 영화들이 나온다
print(get_recommendations('Star Voyage'))

`Star Voyage`를 입력하면 같은 우주 묶음인 `Space Odyssey`, `Galaxy Quest`가 위로 올라온다. 줄거리(overview)의 단어 분포가 비슷할수록 TF-IDF 벡터의 방향이 가까워지고, 코사인 유사도가 높게 나오기 때문이다. 교안의 예시에서 다크 나이트 라이즈를 넣으면 다른 배트맨 영화들이 추천되던 것과 정확히 같은 원리다.

# 5. 그 외의 유사도 측정 방법

문서 유사도를 구하는 방법은 코사인 유사도 말고도 여럿이다. 이 절에서는 **유클리드 거리**와 **자카드 유사도**를 본다. 코사인 유사도만큼 자주 쓰이진 않지만, 다른 개념을 이해하는 데 도움이 된다.

## 5-1. 유클리드 거리 (Euclidean Distance)

다차원 공간의 두 점 $p = (p_1, \ldots, p_n)$ 과 $q = (q_1, \ldots, q_n)$ 사이의 직선 거리를 구하는 공식이 **유클리드 거리**다.

$$ d(p, q) = \sqrt{(q_1 - p_1)^2 + (q_2 - p_2)^2 + \cdots + (q_n - p_n)^2} = \sqrt{\sum_{i=1}^{n}(q_i - p_i)^2} $$

2차원에서는 곧 **피타고라스의 정리**와 같다. 여러 문서의 유사도를 잰다는 것은, 이 2차원을 **단어 개수만큼의 차원**으로 확장하는 것과 같다. 다음 DTM(단어 4개 → 4차원)에서 문서Q와 가장 가까운 문서를 찾아본다.

| | 바나나 | 사과 | 저는 | 좋아요 |
|:--|:--:|:--:|:--:|:--:|
| **문서1** | 2 | 3 | 0 | 1 |
| **문서2** | 1 | 2 | 3 | 1 |
| **문서3** | 2 | 1 | 2 | 2 |
| **문서Q** | 1 | 1 | 0 | 1 |

**거리가 가장 작다 = 가장 가깝다 = 가장 유사하다**는 점에 유의한다(코사인 유사도와 방향이 반대다).

In [ ]:
import numpy as np

def dist(x, y):
    return np.sqrt(np.sum((x - y) ** 2))

doc1 = np.array((2, 3, 0, 1))
doc2 = np.array((1, 2, 3, 1))
doc3 = np.array((2, 1, 2, 2))
docQ = np.array((1, 1, 0, 1))

print('문서1 vs 문서Q :', dist(doc1, docQ))
print('문서2 vs 문서Q :', dist(doc2, docQ))
print('문서3 vs 문서Q :', dist(doc3, docQ))
# 거리가 가장 작은 문서1이 문서Q와 가장 유사하다

## 5-2. 자카드 유사도 (Jaccard Similarity)

**자카드 유사도**는 두 집합의 **합집합 대비 교집합의 비율**로 유사도를 잰다. 두 집합이 같으면 1, 공통 원소가 전혀 없으면 0이며, 항상 **0과 1 사이**의 값을 가진다.

$$ J(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|} $$

두 문서를 단어 집합으로 보고, 공통 단어가 전체 단어 중 얼마나 되는지를 계산한다.

In [ ]:
doc1 = "apple banana everyone like likey watch card holder"
doc2 = "apple banana coupon passport love you"

# 토큰화
tokenized_doc1 = doc1.split()
tokenized_doc2 = doc2.split()

union = set(tokenized_doc1).union(set(tokenized_doc2))         # 합집합
intersection = set(tokenized_doc1).intersection(set(tokenized_doc2))  # 교집합

print('합집합 크기 :', len(union))           # 12
print('교집합      :', intersection)          # {apple, banana}
print('자카드 유사도:', len(intersection) / len(union))   # 2 / 12

---

이로써 11장 **텍스트의 유사도**를 정리했다. 단어를 숫자로 표현하는 두 갈래(국소·분산)에서 출발해, 빈도수 기반의 **Bag of Words → DTM → TF-IDF**로 문서를 벡터화하는 흐름을 익혔다. 그리고 이렇게 만든 벡터로 문서 간 유사도를 재는 **코사인 유사도**(방향 기반, 길이에 강건), **유클리드 거리**(직선 거리, 작을수록 유사), **자카드 유사도**(집합의 교집합/합집합)까지 직접 구현하며 확인했다.

핵심은 **"문서를 벡터로 바꾼 뒤, 벡터 간 유사도를 잰다"**는 하나의 줄기다. 다만 BoW·DTM·TF-IDF는 모두 **단어의 의미·뉘앙스를 담지 못하는 국소(이산) 표현**이고, 차원이 큰 **희소 벡터**라는 한계가 있다. 다음 장부터는 이 한계를 넘어, 단어의 의미를 조밀한(dense) 벡터에 담는 **워드 임베딩(Word Embedding)**과 **Word2Vec**을 본격적으로 다룬다.